In [24]:
import pandas as pd
import numpy as np
from pathlib import Path

# INPUT / OUTPUT
INPUT_CSV = Path("data/MASTER_VARIABLES.csv")
OUTPUT_CSV = Path("data/hazard.csv")

df = pd.read_csv(INPUT_CSV)

# ---------------------------------------------------------
# 1. DISTRICT-MONTH MEAN
# ---------------------------------------------------------
district_stats = (
    df.groupby(["district", "timeperiod"], as_index=False)
      .agg(district_mean_heatday=("mean_heatday", "mean"))
)

# ---------------------------------------------------------
# 2. MONTHLY Z-SCORE
# ---------------------------------------------------------
district_stats["heat_zscore"] = (
    district_stats.groupby("timeperiod")["district_mean_heatday"]
    .transform(lambda x: (x - x.mean()) / x.std(ddof=0) if x.std(ddof=0) != 0 else 0)
)

# ---------------------------------------------------------
# 3. BINNING (1–5)
# ---------------------------------------------------------
def classify(z):
    if z <= -1.5:
        return 1
    elif z <= -0.5:
        return 2
    elif z <= 0.5:
        return 3
    elif z <= 1.5:
        return 4
    else:
        return 5

district_stats["heat_hazard"] = district_stats["heat_zscore"].apply(classify)

# ---------------------------------------------------------
# 4. SAVE ONLY DISTRICT-MONTH OUTPUT (CLEAN)
# ---------------------------------------------------------
district_stats.to_csv(OUTPUT_CSV, index=False)

print(f"Saved clean hazard file: {OUTPUT_CSV}")
print(district_stats.head())

Saved clean hazard file: data/hazard.csv
  district timeperiod  district_mean_heatday  heat_zscore  heat_hazard
0   Anugul    2023_01              11.563157    -0.021585            3
1   Anugul    2023_02              11.563157    -0.021585            3
2   Anugul    2023_03              11.563157    -0.021585            3
3   Anugul    2023_04              32.145668    -0.099194            3
4   Anugul    2023_05              32.145668    -0.099194            3


In [23]:
import pandas as pd
import geopandas as gpd
import numpy as np
import folium
from folium.plugins import TimeSliderChoropleth
import json

# ----------------------------------------------------
# INPUT
# ----------------------------------------------------
hazard_csv = Path("data/hazard.csv")
geojson_path = Path("../Maps/od_ids-drr_shapefiles/odisha_district_final.geojson")

df = pd.read_csv(hazard_csv)
gdf = gpd.read_file(geojson_path)

gdf = gdf.rename(columns={"dtname": "district"})
gdf = gdf.to_crs(epsg=4326)

# ----------------------------------------------------
# CREATE TIME INDEX (convert months to sortable index)
# ----------------------------------------------------
df["timeperiod"] = df["timeperiod"].astype(str)
df["time_id"] = df["timeperiod"].rank(method="dense").astype(int)

# ----------------------------------------------------
# BUILD BASE GEOJSON STRUCTURE
# ----------------------------------------------------
geojson = json.loads(gdf.to_json())

# mapping: district -> hazard per time
time_lookup = {}

for _, row in df.iterrows():
    key = row["district"]

    if key not in time_lookup:
        time_lookup[key] = {}

    time_lookup[key][row["time_id"]] = row["heat_hazard"]

# ----------------------------------------------------
# STYLE DICT (REQUIRED BY FOLIUM TIMESLIDER)
# ----------------------------------------------------
styledict = {}

for i, feature in enumerate(geojson["features"]):
    district = feature["properties"]["district"]

    styledict[district] = {}

    for t in sorted(df["time_id"].unique()):
        value = time_lookup.get(district, {}).get(t, 0)

        # color mapping
        if value == 1:
            color = "#2c7bb6"
        elif value == 2:
            color = "#abd9e9"
        elif value == 3:
            color = "#ffffbf"
        elif value == 4:
            color = "#fdae61"
        else:
            color = "#d7191c"

        styledict[district][str(t)] = {
            "color": color,
            "opacity": 0.7
        }

# ----------------------------------------------------
# MAP BASE
# ----------------------------------------------------
m = folium.Map(location=[20.5, 84.5], zoom_start=7)

# ----------------------------------------------------
# TIME SLIDER
# ----------------------------------------------------
TimeSliderChoropleth(
    data=geojson,
    styledict=styledict
).add_to(m)

# ----------------------------------------------------
# SAVE
# ----------------------------------------------------
m.save("odisha_heat_hazard_timeslider.html")

print("Saved: odisha_heat_hazard_timeslider.html")

Saved: odisha_heat_hazard_timeslider.html
